# 数据探索与描述性统计

本 notebook 针对 `data/` 下的大规模 Parquet 数据（train ~45GB, test ~12GB）进行：
- **懒加载**：先看 schema、抽样再决定是否全量加载
- **高效统计**：分块/抽样计算描述性统计，为建模提供指导

In [1]:
import os
from pathlib import Path

import pyarrow
import pyarrow.parquet as pq
import pandas as pd

DATA_DIR = Path("data")
TRAIN_PATH = DATA_DIR / "train.parquet"
TEST_PATH = DATA_DIR / "test.parquet"

# 抽样/分块参数：大表先抽样或按行组读，避免 OOM
SAMPLE_ROWS = 100_000   # 初步探索时的抽样行数，可调大
CHUNK_ROWS = 500_000    # 分块统计时的块大小
MAX_MEMORY_MB = 2000    # 单次加载尽量不超过约 2GB（可按机器内存调整）

## 1. 数据结构概览（懒加载：只读元数据）

不加载数据体，仅读取 Parquet 的 schema 与行组信息，了解列名、类型、行数、文件大小。

In [2]:
def get_parquet_meta(path: Path):
    """仅读取 Parquet 元数据，不加载数据体。"""
    if not path.exists():
        return None
    meta = pq.read_metadata(path)
    schema = pq.read_schema(path)  # 单独读 schema，兼容各版本
    return {
        "path": str(path),
        "size_mb": path.stat().st_size / (1024 ** 2),
        "num_rows": meta.num_rows,
        "num_row_groups": meta.num_row_groups,
        "schema": schema,
    }

for name, path in [("train", TRAIN_PATH), ("test", TEST_PATH)]:
    m = get_parquet_meta(path)
    if m:
        print(f"=== {name} ===")
        print(f"  路径: {m['path']}")
        print(f"  文件大小: {m['size_mb']:.1f} MB")
        print(f"  行数: {m['num_rows']:,}")
        print(f"  行组数: {m['num_row_groups']}")
        print(f"  Schema:")
        for f in m["schema"]:
            print(f"    - {f.name}: {f.type}")
        print()

=== train ===
  路径: data\train.parquet
  文件大小: 42654.4 MB
  行数: 43,020,000
  行组数: 500
  Schema:
    - stockid: int32
    - dateid: int32
    - timeid: int32
    - exchangeid: int32
    - f0: float
    - f1: float
    - f2: float
    - f3: float
    - f4: float
    - f5: float
    - f6: float
    - f7: float
    - f8: float
    - f9: float
    - f10: float
    - f11: float
    - f12: float
    - f13: float
    - f14: float
    - f15: float
    - f16: float
    - f17: float
    - f18: float
    - f19: float
    - f20: float
    - f21: float
    - f22: float
    - f23: float
    - f24: float
    - f25: float
    - f26: float
    - f27: float
    - f28: float
    - f29: float
    - f30: float
    - f31: float
    - f32: float
    - f33: float
    - f34: float
    - f35: float
    - f36: float
    - f37: float
    - f38: float
    - f39: float
    - f40: float
    - f41: float
    - f42: float
    - f43: float
    - f44: float
    - f45: float
    - f46: float
    - f47: float
    - f48: fl

## 2. 抽样查看数据（懒加载：只读前若干行）

用 PyArrow 按行读取前 N 行，转成 pandas 便于查看；避免一次性读入整张表。

In [3]:
def read_head(path: Path, n: int = 5) -> pd.DataFrame:
    """只读前 n 行：按行组读，避免整表加载。"""
    pf = pq.ParquetFile(path)
    chunks = []
    got = 0
    for i in range(pf.metadata.num_row_groups):
        if got >= n:
            break
        table = pf.read_row_group(i)
        chunks.append(table)
        got += table.num_rows
    if not chunks:
        return pd.DataFrame()
    table = pyarrow.concat_tables(chunks)
    return table.slice(0, n).to_pandas()

# 先看 train 前 5 行
df_head = read_head(TRAIN_PATH, 5)
print("Train 前 5 行:")
display(df_head)
print("\n列 dtypes:")
print(df_head.dtypes)

Train 前 5 行:


,stockid,dateid,timeid,exchangeid,f0,f1,f2,f3,f4,f5,...,f377,f378,f379,f380,f381,f382,f383,LabelA,LabelB,LabelC
0,0,0,0,0,0.000000,0.827602,NaN,NaN,8.091544e-39,1.921693,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.023960,-0.033270,-0.028307
1,0,0,1,0,0.170141,1.502839,1.036712,-0.270918,1.618461e-01,0.794303,...,0.599513,0.831422,0.826040,-0.030138,-0.173462,0.265666,0.606023,-0.018115,-0.024315,-0.021936
2,0,0,2,0,0.097716,2.475088,1.390699,0.184700,1.326206e-01,0.897420,...,0.739451,0.484083,0.755341,-0.075283,-0.016207,-0.240854,-0.061096,-0.008411,-0.010968,-0.018479
3,0,0,3,0,0.095084,1.809949,0.501454,0.023560,1.553270e-01,2.266376,...,0.723045,0.619431,0.580260,0.230448,-0.140937,0.097294,-0.088349,-0.011146,-0.016765,-0.022597
4,0,0,4,0,0.087559,1.863069,1.077359,0.259082,2.944657e-01,1.015742,...,0.658806,0.634629,0.747044,0.146748,0.031036,0.330634,0.257386,-0.012444,-0.015333,-0.019900



列 dtypes:
stockid         int32
dateid          int32
timeid          int32
exchangeid      int32
f0            float32
               ...   
f382          float32
f383          float32
LabelA        float32
LabelB        float32
LabelC        float32
Length: 391, dtype: object


In [4]:
# 可选：同样方式看 test 前几行
df_test_head = read_head(TEST_PATH, 5)
print("Test 前 5 行:")
display(df_test_head)

Test 前 5 行:


,stockid,dateid,timeid,exchangeid,f0,f1,f2,f3,f4,f5,...,f374,f375,f376,f377,f378,f379,f380,f381,f382,f383
0,0,0,0,0,NaN,NaN,NaN,NaN,0.000000e+00,0.005750,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,0,1,0,0.145472,1.068435,1.118942,-1.185533,5.408017e-01,0.439508,...,0.629473,0.543686,0.934621,0.980158,0.950169,0.923248,0.932335,-0.124619,0.945672,0.640437
2,0,0,2,0,0.124288,0.966545,0.928851,1.129364,6.422996e-01,0.409538,...,-0.375687,0.401356,1.000000,0.980481,0.940989,0.789477,-0.212938,0.015838,0.845466,0.127151
3,0,0,3,0,0.045049,0.000000,NaN,NaN,3.097762e-08,0.000000,...,0.243389,0.426795,1.000000,1.000000,0.922872,1.000000,0.652758,0.088726,-0.052632,-0.120246
4,0,0,4,0,0.067881,0.000000,NaN,NaN,2.825049e-35,0.404762,...,-0.022080,0.334994,0.993288,0.989318,0.927998,0.895154,0.783663,-0.366428,-0.098798,0.169355


## 3. 描述性统计（抽样版，快速）

对 train 做行抽样后做 describe + 缺失/唯一值，内存可控；适合先快速看一版。

In [5]:
def read_sample_by_row_groups(path: Path, max_rows: int):
    """按行组顺序读，最多读满 max_rows 行，避免全量加载。"""
    pf = pq.ParquetFile(path)
    chunks = []
    got = 0
    for i in range(pf.metadata.num_row_groups):
        if got >= max_rows:
            break
        table = pf.read_row_group(i)
        if got + table.num_rows > max_rows:
            table = table.slice(0, max_rows - got)
        chunks.append(table)
        got += table.num_rows
    if not chunks:
        return pd.DataFrame()
    return pyarrow.concat_tables(chunks).to_pandas()

# 抽样行数可调：机器内存大可适当增大 SAMPLE_ROWS
df_sample = read_sample_by_row_groups(TRAIN_PATH, SAMPLE_ROWS)
print(f"抽样行数: {len(df_sample):,}")
print("\n--- 数值列 describe ---")
display(df_sample.describe())

抽样行数: 100,000

--- 数值列 describe ---


,stockid,dateid,timeid,exchangeid,f0,f1,f2,f3,f4,f5,...,f377,f378,f379,f380,f381,f382,f383,LabelA,LabelB,LabelC
count,100000.000000,100000.000000,100000.00000,100000.0,7.746700e+04,77467.000000,43460.000000,43460.000000,7.502600e+04,75026.000000,...,77167.000000,92308.000000,91835.000000,74227.000000,73506.000000,64057.000000,86850.000000,100000.000000,100000.000000,100000.000000
mean,0.139600,158.449310,118.93091,0.0,7.196896e-02,0.695997,0.643210,0.037942,7.058877e-02,0.643250,...,0.847758,0.794228,0.784712,0.096727,0.115956,0.118320,0.087394,-0.000013,-0.000058,-0.000137
std,0.346573,109.832724,69.00024,0.0,1.496695e-01,0.729763,0.248907,0.787333,1.444106e-01,0.682240,...,0.183740,0.200357,0.213179,0.301147,0.311922,0.327024,0.295454,0.006417,0.010108,0.013085
min,0.000000,0.000000,0.00000,0.0,0.000000e+00,-3.218841,0.000247,-5.258777,0.000000e+00,-2.421950,...,-0.278454,-0.331485,-0.308405,-0.999332,-1.000000,-0.520494,-0.553313,-0.073361,-0.086985,-0.086515
25%,0.000000,52.000000,59.00000,0.0,5.984170e-19,0.170909,0.496810,-0.259269,6.110839e-16,0.131266,...,0.755783,0.684781,0.664238,-0.096396,-0.090998,-0.092453,-0.104753,-0.002861,-0.004618,-0.006076
50%,0.000000,150.000000,119.00000,0.0,2.363964e-04,0.574883,0.641789,0.058257,6.242079e-04,0.544354,...,0.917815,0.846590,0.842670,-0.052632,-0.037700,-0.052632,-0.045188,-0.000216,-0.000446,-0.000698
75%,0.000000,255.000000,179.00000,0.0,7.730474e-02,0.957253,0.801412,0.371508,7.843914e-02,0.892466,...,1.000000,0.961074,0.966988,0.217905,0.258604,0.245639,0.206559,0.002380,0.003491,0.004204
max,1.000000,359.000000,238.00000,0.0,9.900498e-01,8.996834,1.652780,5.916506,9.900498e-01,10.564973,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.117583,0.159703,0.165617


In [6]:
# 缺失值、唯一值（抽样）
print("--- 缺失数 / 缺失率 ---")
missing = df_sample.isnull().sum()
missing_pct = (missing / len(df_sample) * 100).round(2)
missing_df = pd.DataFrame({"缺失数": missing, "缺失率%": missing_pct})
display(missing_df[missing_df["缺失数"] > 0].sort_values("缺失数", ascending=False))

print("\n--- 部分列唯一值数量（抽样）---")
nunique = df_sample.nunique()
# 只显示唯一值较少的列，便于判断类别型
low_card = nunique[nunique <= 50].sort_values()
if len(low_card) > 0:
    display(pd.DataFrame({"唯一值数": low_card}))
else:
    print("(抽样中无低基数列，可对关键列单独 nunique)")

--- 缺失数 / 缺失率 ---


,缺失数,缺失率%
f200,100000,100.00
f197,100000,100.00
f199,100000,100.00
f231,100000,100.00
f296,100000,100.00
...,...,...
f122,143,0.14
f174,91,0.09
f181,91,0.09
f182,91,0.09



--- 部分列唯一值数量（抽样）---


,唯一值数
f200,0
f293,0
f234,0
f233,0
f232,0
f231,0
f296,0
f294,0
f199,0
f197,0


## 4. 全量描述性统计（分块遍历，完整运行用）

需要全表统计时，按行组分块读入，增量计算 count/sum/sum2 等，最后汇总均值、标准差，避免一次性加载整表。

In [7]:
def chunked_numeric_stats(path: Path, numeric_columns: list = None):
    """
    分块遍历 Parquet，对数值列按列计算非空 count/sum/sum_sq/min/max，再算 mean/std。
    numeric_columns 为 None 时自动只选数值类型列。
    返回每列的 count, mean, std, min, max（即取值范围）。
    """
    pf = pq.ParquetFile(path)
    schema = pf.schema_arrow
    if numeric_columns is None:
        numeric_columns = [f.name for f in schema if pyarrow.types.is_floating(f.type) or pyarrow.types.is_integer(f.type)]
    if not numeric_columns:
        return pd.DataFrame()

    counts = {c: 0 for c in numeric_columns}
    sums = {c: 0.0 for c in numeric_columns}
    sum_sq = {c: 0.0 for c in numeric_columns}
    mins = {c: None for c in numeric_columns}   # 取值范围：最小值
    maxs = {c: None for c in numeric_columns}  # 取值范围：最大值

    for i in range(pf.metadata.num_row_groups):
        table = pf.read_row_group(i)
        df = table.to_pandas()
        cols = [c for c in numeric_columns if c in df.columns]
        df = df[cols].astype(float, errors="ignore").select_dtypes(include=["number"])
        if df.empty:
            continue
        for c in df.columns:
            valid = df[c].dropna()
            if len(valid) == 0:
                continue
            counts[c] = counts.get(c, 0) + len(valid)
            sums[c] = sums.get(c, 0) + valid.sum()
            sum_sq[c] = sum_sq.get(c, 0) + (valid ** 2).sum()
            c_min, c_max = valid.min(), valid.max()
            mins[c] = c_min if mins[c] is None else min(mins[c], c_min)
            maxs[c] = c_max if maxs[c] is None else max(maxs[c], c_max)

    rows = []
    for c in numeric_columns:
        n = counts.get(c, 0)
        if n == 0:
            continue
        mean = sums[c] / n
        var = (sum_sq[c] / n) - (mean ** 2)
        std = (var ** 0.5) if var >= 0 else 0.0
        rows.append({"col": c, "count": n, "mean": mean, "std": std, "min": mins[c], "max": maxs[c]})
    return pd.DataFrame(rows)

# 仅当需要全量统计时取消注释并运行（耗时会较长）
full_stats = chunked_numeric_stats(TRAIN_PATH)
display(full_stats)

,col,count,mean,std,min,max
0,stockid,43020000,249.500000,144.337279,0.000000,499.000000
1,dateid,43020000,179.500000,103.922648,0.000000,359.000000
2,timeid,43020000,119.000000,68.992753,0.000000,238.000000
3,exchangeid,43020000,0.552000,0.497289,0.000000,1.000000
4,f0,40298880,0.067802,0.109639,0.000000,0.990050
...,...,...,...,...,...,...
386,f382,37599547,0.078889,0.289987,-1.000000,1.000000
387,f383,41846208,0.061441,0.274797,-1.000000,1.000000
388,LabelA,43020000,0.000018,0.005338,-0.683097,0.683097
389,LabelB,43020000,0.000032,0.008283,-0.229893,0.683097


In [8]:
# 根据 full_stats 判断：count == stockid 的 count 即认为该特征无缺失
stockid_count = full_stats.loc[full_stats["col"] == "stockid", "count"].squeeze()
no_missing = full_stats[full_stats["count"] == stockid_count]
no_missing_cols = no_missing["col"].tolist()

print(f"总行数（以 stockid 为准）: {stockid_count:,}")
print(f"无缺失特征数: {len(no_missing_cols)}")
print("\n无缺失特征列表:")
print(no_missing_cols)
display(no_missing)

总行数（以 stockid 为准）: 43,020,000
无缺失特征数: 94

无缺失特征列表:
['stockid', 'dateid', 'timeid', 'exchangeid', 'f29', 'f41', 'f46', 'f47', 'f48', 'f58', 'f59', 'f61', 'f170', 'f171', 'f172', 'f173', 'f219', 'f223', 'f230', 'f235', 'f236', 'f237', 'f238', 'f258', 'f259', 'f260', 'f261', 'f262', 'f263', 'f264', 'f265', 'f266', 'f270', 'f273', 'f274', 'f279', 'f280', 'f281', 'f282', 'f283', 'f286', 'f287', 'f290', 'f292', 'f297', 'f298', 'f299', 'f300', 'f313', 'f314', 'f315', 'f316', 'f317', 'f318', 'f319', 'f320', 'f321', 'f322', 'f323', 'f324', 'f325', 'f334', 'f335', 'f336', 'f338', 'f339', 'f340', 'f341', 'f342', 'f343', 'f344', 'f345', 'f346', 'f350', 'f354', 'f355', 'f356', 'f357', 'f358', 'f359', 'f360', 'f361', 'f362', 'f366', 'f367', 'f368', 'f369', 'f370', 'f371', 'f372', 'f373', 'LabelA', 'LabelB', 'LabelC']


,col,count,mean,std,min,max
0,stockid,43020000,249.500000,144.337279,0.000000,499.000000
1,dateid,43020000,179.500000,103.922648,0.000000,359.000000
2,timeid,43020000,119.000000,68.992753,0.000000,238.000000
3,exchangeid,43020000,0.552000,0.497289,0.000000,1.000000
33,f29,43020000,-0.152612,0.384101,-9.144825,6.353186
...,...,...,...,...,...,...
376,f372,43020000,1.474897,1.121878,0.000000,9.055673
377,f373,43020000,0.590861,0.687576,0.000000,5.356586
388,LabelA,43020000,0.000018,0.005338,-0.683097,0.683097
389,LabelB,43020000,0.000032,0.008283,-0.229893,0.683097


In [9]:
full_stats_test = chunked_numeric_stats(TEST_PATH)
display(full_stats_test)

,col,count,mean,std
0,stockid,11352500,249.500000,144.337279
1,dateid,11352500,47.000000,27.422618
2,timeid,11352500,119.000000,68.992753
3,exchangeid,11352500,0.552000,0.497289
4,f0,10709686,0.066935,0.114632
...,...,...,...,...
383,f379,11160706,0.625830,0.249017
384,f380,10439196,0.087935,0.292574
385,f381,10261258,0.092708,0.292624
386,f382,10040034,0.094352,0.299655


In [10]:
# 测试集同样判断：count == stockid 的 count 即认为该特征无缺失
stockid_count_test = full_stats_test.loc[full_stats_test["col"] == "stockid", "count"].squeeze()
no_missing_test = full_stats_test[full_stats_test["count"] == stockid_count_test]
no_missing_cols_test = no_missing_test["col"].tolist()

print(f"测试集总行数（以 stockid 为准）: {stockid_count_test:,}")
print(f"测试集无缺失特征数: {len(no_missing_cols_test)}")
print("\n测试集无缺失特征列表:")
print(no_missing_cols_test)
display(no_missing_test)

测试集总行数（以 stockid 为准）: 11,352,500
测试集无缺失特征数: 91

测试集无缺失特征列表:
['stockid', 'dateid', 'timeid', 'exchangeid', 'f29', 'f41', 'f46', 'f47', 'f48', 'f58', 'f59', 'f61', 'f170', 'f171', 'f172', 'f173', 'f219', 'f223', 'f230', 'f235', 'f236', 'f237', 'f238', 'f258', 'f259', 'f260', 'f261', 'f262', 'f263', 'f264', 'f265', 'f266', 'f270', 'f273', 'f274', 'f279', 'f280', 'f281', 'f282', 'f283', 'f286', 'f287', 'f290', 'f292', 'f297', 'f298', 'f299', 'f300', 'f313', 'f314', 'f315', 'f316', 'f317', 'f318', 'f319', 'f320', 'f321', 'f322', 'f323', 'f324', 'f325', 'f334', 'f335', 'f336', 'f338', 'f339', 'f340', 'f341', 'f342', 'f343', 'f344', 'f345', 'f346', 'f350', 'f354', 'f355', 'f356', 'f357', 'f358', 'f359', 'f360', 'f361', 'f362', 'f366', 'f367', 'f368', 'f369', 'f370', 'f371', 'f372', 'f373']


,col,count,mean,std
0,stockid,11352500,249.500000,144.337279
1,dateid,11352500,47.000000,27.422618
2,timeid,11352500,119.000000,68.992753
3,exchangeid,11352500,0.552000,0.497289
33,f29,11352500,-0.165587,0.403014
...,...,...,...,...
373,f369,11352500,3.487474,3.849437
374,f370,11352500,1.444796,2.903748
375,f371,11352500,1.366567,2.855452
376,f372,11352500,1.388019,1.080737


## 5. 建模简要建议

根据上述探索可进一步：
- **目标列**：从 schema 与业务含义确定标签列，检查分布与缺失。
- **特征类型**：数值列做标准化/分桶；低基数列做类别编码或嵌入。
- **缺失**：按缺失率与业务决定删除/插补/单独标记。
- **规模**：全量训练时建议用 Dask 或分块/增量训练，避免单机 OOM。

### 因子 IC 分析（先算截面 IC，再求 IC 均值 & ICIR）

- **按 dateid 分组**：每个日期内算因子与 label 的截面 IC → 得到 IC 时序 → IC均值 / IC标准差 / ICIR
- **按 stockid 分组**：每个股票内算 IC → IC均值 / IC标准差 / ICIR
- **按 (dateid, stockid) 分组**：每个 (日期, 股票) 内算 IC → IC均值 / IC标准差 / ICIR

其中 ICIR = IC均值 / IC标准差

In [18]:
import numpy as np
import pandas as pd
from typing import List, Dict
from pathlib import Path
import pyarrow
import pyarrow.parquet as pq
import time


def compute_ic_analysis(
    path: Path,
    factor_cols: List[str],
    label_cols: List[str],
    do_overall: bool = True,
    do_by_date: bool = True,
    do_by_stock: bool = True,
    do_by_date_stock: bool = False,
) -> Dict[str, pd.DataFrame]:
    """先在各分组内算截面 IC（Pearson 相关），再对 IC 求均值 / 标准差 / ICIR。

    逻辑（以 by_date 为例）：
      1) 对每个 dateid，在该日期的所有行上算 corr(factor, label) = 该日截面 IC
      2) 收集所有日期的 IC → IC 时序
      3) IC均值 = mean(IC序列), IC标准差 = std(IC序列), ICIR = IC均值 / IC标准差

    高效实现：
      - 按 Parquet 行组分块读取，不一次性载入全表
      - 对每个分组维护增量统计量 (n, Σx, Σx², Σy, Σy², Σxy)
      - 最终一次性向量化计算全部组的 IC，再做聚合

    Parameters
    ----------
    path : Parquet 文件路径
    factor_cols : 因子列名列表
    label_cols : 目标变量列名列表
    do_overall : 全样本 IC（不分组），默认开启
    do_by_date : 按 dateid 分组（截面 IC 时序），默认开启
    do_by_stock : 按 stockid 分组，默认开启
    do_by_date_stock : 按 (dateid, stockid) 分组（组数多，默认关闭）

    Returns
    -------
    dict，键为 'overall' / 'by_date' / 'by_stock' / 'by_date_stock'（取决于开关）。
    - overall: DataFrame，列为 factor, label, ic, n（全样本单个 IC 值）
    - 其余: DataFrame，列为 factor, label, ic_mean, ic_std, icir, n_groups
    """
    pf = pq.ParquetFile(path)
    nf = len(factor_cols)
    nl = len(label_cols)
    n_rg = pf.metadata.num_row_groups
    read_cols = ["dateid", "stockid"] + factor_cols + label_cols

    # ---- 全样本 overall 增量统计（不分组） ----
    if do_overall:
        ov_n   = np.zeros(nf)
        ov_sx  = np.zeros(nf)
        ov_sx2 = np.zeros(nf)
        ov_sy  = np.zeros((nf, nl))
        ov_sy2 = np.zeros((nf, nl))
        ov_sxy = np.zeros((nf, nl))

    # ---- 存储：每种分组一个 dict, key → (n, sx, sx2, sy, sy2, sxy) ----
    # n/sx/sx2: shape (nf,)    sy/sy2/sxy: shape (nf, nl)
    stores: Dict[str, dict] = {}
    if do_by_date:
        stores["by_date"] = {}
    if do_by_stock:
        stores["by_stock"] = {}
    if do_by_date_stock:
        stores["by_date_stock"] = {}

    def _accum(store, key, M_g, Fc_g, L_g):
        """增量累加一个子组的统计量（全部向量化，无 factor/label 循环）。
        M_g : (n, nf) bool   — 因子有限值掩码
        Fc_g: (n, nf) float  — 因子（NaN 置零）
        L_g : (n, nl) float  — 标签（假设无 NaN）
        """
        Mf = M_g.astype(np.float64)
        inc = (
            M_g.sum(axis=0),            # n    (nf,)
            Fc_g.sum(axis=0),           # sx   (nf,)
            (Fc_g ** 2).sum(axis=0),    # sx2  (nf,)
            Mf.T @ L_g,                # sy   (nf, nl) — 只对因子有效行求和
            Mf.T @ (L_g ** 2),          # sy2  (nf, nl)
            Fc_g.T @ L_g,              # sxy  (nf, nl)
        )
        if key not in store:
            store[key] = tuple(a.copy() for a in inc)
        else:
            p = store[key]
            store[key] = tuple(p[j] + inc[j] for j in range(6))

    # ---- 主循环：逐行组读取 ----
    t0 = time.time()
    for rg in range(n_rg):
        if rg % 50 == 0:
            elapsed = time.time() - t0
            print(f"  行组 {rg}/{n_rg}  ({elapsed:.1f}s)")
        table = pf.read_row_group(rg, columns=read_cols)
        df = table.to_pandas()

        date_arr = df["dateid"].to_numpy()
        stock_arr = df["stockid"].to_numpy()
        F = df[factor_cols].to_numpy(dtype="float64")
        L = df[label_cols].to_numpy(dtype="float64")
        M = np.isfinite(F)
        Fc = np.where(M, F, 0.0)

        # ---- overall 累加（不分组） ----
        if do_overall:
            Mf = M.astype(np.float64)
            ov_n   += M.sum(axis=0)
            ov_sx  += Fc.sum(axis=0)
            ov_sx2 += (Fc ** 2).sum(axis=0)
            ov_sy  += Mf.T @ L
            ov_sy2 += Mf.T @ (L ** 2)
            ov_sxy += Fc.T @ L

        if do_by_date:
            for d in np.unique(date_arr):
                idx = date_arr == d
                _accum(stores["by_date"], int(d), M[idx], Fc[idx], L[idx])

        if do_by_stock:
            for s in np.unique(stock_arr):
                idx = stock_arr == s
                _accum(stores["by_stock"], int(s), M[idx], Fc[idx], L[idx])

        if do_by_date_stock:
            ck = date_arr.astype(np.int64) * 100000 + stock_arr.astype(np.int64)
            for c in np.unique(ck):
                idx = ck == c
                _accum(stores["by_date_stock"], int(c),
                       M[idx], Fc[idx], L[idx])

    print(f"  扫描完成，共 {n_rg} 行组，用时 {time.time()-t0:.1f}s")

    # ---- 汇总：从增量统计 → 每组 IC → IC 均值 / 标准差 / ICIR ----
    def _summarize(store) -> pd.DataFrame:
        keys = list(store.keys())
        G = len(keys)
        if G == 0:
            return pd.DataFrame(
                columns=["factor", "label", "ic_mean", "ic_std", "icir", "n_groups"]
            )

        all_n   = np.stack([store[k][0] for k in keys])    # (G, nf)
        all_sx  = np.stack([store[k][1] for k in keys])
        all_sx2 = np.stack([store[k][2] for k in keys])
        all_sy  = np.stack([store[k][3] for k in keys])    # (G, nf, nl)
        all_sy2 = np.stack([store[k][4] for k in keys])
        all_sxy = np.stack([store[k][5] for k in keys])

        with np.errstate(divide="ignore", invalid="ignore"):
            mean_x = all_sx / all_n                             # (G, nf)
            var_x  = all_sx2 / all_n - mean_x ** 2

            n3   = all_n[:, :, np.newaxis]                      # (G, nf, 1)
            mx3  = mean_x[:, :, np.newaxis]
            vx3  = var_x[:, :, np.newaxis]

            mean_y = all_sy / n3                                # (G, nf, nl)
            var_y  = all_sy2 / n3 - mean_y ** 2
            cov_xy = all_sxy / n3 - mx3 * mean_y

            denom = np.sqrt(np.maximum(vx3, 0) * np.maximum(var_y, 0))
            ok = (denom > 0) & (n3 > 1)
            ic = np.where(ok, cov_xy / denom, np.nan)          # (G, nf, nl)

        # IC 均值、标准差、ICIR（跨组聚合）
        ic_mean = np.nanmean(ic, axis=0)                        # (nf, nl)
        ic_std  = np.nanstd(ic, axis=0, ddof=1)
        with np.errstate(divide="ignore", invalid="ignore"):
            icir = np.where(ic_std > 0, ic_mean / ic_std, np.nan)
        n_valid = np.sum(np.isfinite(ic), axis=0).astype(int)

        rows = []
        for fi in range(nf):
            for li in range(nl):
                rows.append({
                    "factor": factor_cols[fi],
                    "label":  label_cols[li],
                    "ic_mean": ic_mean[fi, li],
                    "ic_std":  ic_std[fi, li],
                    "icir":    icir[fi, li],
                    "n_groups": n_valid[fi, li],
                })
        return pd.DataFrame(rows)

    results = {}

    # ---- overall: 从全局累加器算一个 IC 值 ----
    if do_overall:
        print("  汇总 overall...")
        ov_rows = []
        with np.errstate(divide="ignore", invalid="ignore"):
            mean_x = ov_sx / ov_n                                   # (nf,)
            var_x  = ov_sx2 / ov_n - mean_x ** 2
            n3  = ov_n[:, np.newaxis]                                # (nf, 1)
            mx3 = mean_x[:, np.newaxis]
            vx3 = var_x[:, np.newaxis]
            mean_y = ov_sy / n3                                      # (nf, nl)
            var_y  = ov_sy2 / n3 - mean_y ** 2
            cov_xy = ov_sxy / n3 - mx3 * mean_y
            denom  = np.sqrt(np.maximum(vx3, 0) * np.maximum(var_y, 0))
            ok     = (denom > 0) & (n3 > 1)
            ic_all = np.where(ok, cov_xy / denom, np.nan)           # (nf, nl)
        for fi in range(nf):
            for li in range(nl):
                ov_rows.append({
                    "factor": factor_cols[fi],
                    "label":  label_cols[li],
                    "ic":     ic_all[fi, li],
                    "n":      int(ov_n[fi]),
                })
        results["overall"] = pd.DataFrame(ov_rows)

    for name, store in stores.items():
        print(f"  汇总 {name} ({len(store)} 组)...")
        results[name] = _summarize(store)

    print(f"全部完成，总用时 {time.time()-t0:.1f}s")
    return results

In [19]:
# ====== 调用示例 ======
TRAIN_PATH = Path(r"data\train.parquet")
factor_cols = [f"f{i}" for i in range(383)]
label_cols = ["LabelA", "LabelB", "LabelC"]

# do_by_date_stock 默认关闭，组数多（~47500）、耗时较长；需要时手动开启
ic_result = compute_ic_analysis(
    TRAIN_PATH,
    factor_cols=factor_cols,
    label_cols=label_cols,
    do_overall=True,
    do_by_date=True,
    do_by_stock=True,
    do_by_date_stock=False,
)

print("\n=== 全样本 overall IC ===")
display(ic_result["overall"].sort_values("ic", key=abs, ascending=False).head(20))

print("\n=== 按 dateid 分组的 IC 分析（截面 IC 时序）===")
display(ic_result["by_date"].sort_values("icir", ascending=False).head(20))

print("\n=== 按 stockid 分组的 IC 分析 ===")
display(ic_result["by_stock"].sort_values("icir", ascending=False).head(20))

  行组 0/500  (0.0s)
  行组 50/500  (105.9s)
  行组 100/500  (213.2s)
  行组 150/500  (317.2s)
  行组 200/500  (423.8s)
  行组 250/500  (527.4s)
  行组 300/500  (630.9s)
  行组 350/500  (738.2s)
  行组 400/500  (842.4s)
  行组 450/500  (947.5s)
  扫描完成，共 500 行组，用时 1051.4s
  汇总 overall...
  汇总 by_date (360 组)...
  汇总 by_stock (500 组)...
全部完成，总用时 1051.5s

=== 全样本 overall IC ===


,factor,label,ic,n
894,f298,LabelA,0.036383,43020000
975,f325,LabelA,-0.030291,43020000
972,f324,LabelA,0.027812,43020000
895,f298,LabelB,0.027666,43020000
855,f285,LabelA,0.027576,40390243
489,f163,LabelA,0.027156,23542427
927,f309,LabelA,-0.026911,42840000
543,f181,LabelA,0.025850,43015486
976,f325,LabelB,-0.023969,43020000
186,f62,LabelA,0.023708,41560302



=== 按 dateid 分组的 IC 分析（截面 IC 时序）===


,factor,label,ic_mean,ic_std,icir,n_groups
855,f285,LabelA,0.029373,0.008511,3.451212,360
108,f36,LabelA,0.016672,0.006136,2.717016,360
801,f267,LabelA,0.008864,0.003322,2.668321,360
687,f229,LabelA,0.019128,0.008104,2.360390,360
387,f129,LabelA,0.009423,0.004083,2.308038,360
894,f298,LabelA,0.034046,0.016025,2.124604,360
856,f285,LabelB,0.023998,0.011527,2.081899,360
804,f268,LabelA,0.006512,0.003217,2.024416,360
951,f317,LabelA,0.018920,0.009595,1.971883,360
802,f267,LabelB,0.005704,0.002913,1.957992,360



=== 按 stockid 分组的 IC 分析 ===


,factor,label,ic_mean,ic_std,icir,n_groups
723,f241,LabelA,0.024115,0.007260,3.321776,500
855,f285,LabelA,0.029989,0.009875,3.036898,500
687,f229,LabelA,0.019841,0.006569,3.020320,500
843,f281,LabelA,0.019792,0.007456,2.654545,500
894,f298,LabelA,0.038087,0.015022,2.535449,500
384,f128,LabelA,0.024504,0.009959,2.460575,500
688,f229,LabelB,0.015545,0.006444,2.412250,500
849,f283,LabelA,0.024618,0.010242,2.403697,500
724,f241,LabelB,0.020472,0.008544,2.396198,500
856,f285,LabelB,0.023036,0.009824,2.344807,500


In [22]:
# ====== 保存 IC 分析结果（补充完整版）======
for name, df_ic in ic_result.items():
    out_name = f"ic_analysis_{name}.csv"
    df_ic.to_csv(out_name, index=False)
    print(f"已保存 {out_name} ({len(df_ic)} 行)")
    # 可选：检查保存的文件内容前几行
    print(df_ic.head())

print("全部 IC 结果已保存。")

已保存 ic_analysis_overall.csv (1149 行)
  factor   label        ic         n
0     f0  LabelA -0.009128  40298880
1     f0  LabelB -0.007844  40298880
2     f0  LabelC -0.007262  40298880
3     f1  LabelA  0.010153  40298880
4     f1  LabelB  0.006428  40298880
已保存 ic_analysis_by_date.csv (1149 行)
  factor   label   ic_mean    ic_std      icir  n_groups
0     f0  LabelA -0.008260  0.023982 -0.344416       360
1     f0  LabelB -0.006197  0.037477 -0.165341       360
2     f0  LabelC -0.005769  0.046266 -0.124690       360
3     f1  LabelA  0.014437  0.023158  0.623386       360
4     f1  LabelB  0.014077  0.035236  0.399517       360
已保存 ic_analysis_by_stock.csv (1149 行)
  factor   label   ic_mean    ic_std      icir  n_groups
0     f0  LabelA -0.010585  0.012260 -0.863343       500
1     f0  LabelB -0.009868  0.013415 -0.735589       500
2     f0  LabelC -0.009276  0.013769 -0.673670       500
3     f1  LabelA  0.010531  0.009878  1.066123       500
4     f1  LabelB  0.005963  0.011937  0

In [23]:
# ====== 整合三个 IC 分析表 ======
overall = pd.read_csv("ic_analysis_overall.csv")
by_date = pd.read_csv("ic_analysis_by_date.csv")
by_stock = pd.read_csv("ic_analysis_by_stock.csv")

# 重命名列，避免合并后冲突
overall_tmp = overall.rename(columns={"ic": "ic_overall", "n": "n_overall"})
by_date_tmp = by_date.rename(columns={
    "ic_mean": "ic_mean_by_date", "ic_std": "ic_std_by_date",
    "icir": "icir_by_date", "n_groups": "n_groups_by_date"
})
by_stock_tmp = by_stock.rename(columns={
    "ic_mean": "ic_mean_by_stock", "ic_std": "ic_std_by_stock",
    "icir": "icir_by_stock", "n_groups": "n_groups_by_stock"
})

# 按 factor, label 合并
df_ic_merged = overall_tmp.merge(by_date_tmp, on=["factor", "label"], how="outer")
df_ic_merged = df_ic_merged.merge(by_stock_tmp, on=["factor", "label"], how="outer")

df_ic_merged.to_csv("ic_analysis_merged.csv", index=False)
print(f"已保存 ic_analysis_merged.csv ({len(df_ic_merged)} 行)")
display(df_ic_merged.head(10))

已保存 ic_analysis_merged.csv (1149 行)


,factor,label,ic_overall,n_overall,ic_mean_by_date,ic_std_by_date,icir_by_date,n_groups_by_date,ic_mean_by_stock,ic_std_by_stock,icir_by_stock,n_groups_by_stock
0,f0,LabelA,-0.009128,40298880,-0.008260,0.023982,-0.344416,360,-0.010585,0.012260,-0.863343,500
1,f0,LabelB,-0.007844,40298880,-0.006197,0.037477,-0.165341,360,-0.009868,0.013415,-0.735589,500
2,f0,LabelC,-0.007262,40298880,-0.005769,0.046266,-0.124690,360,-0.009276,0.013769,-0.673670,500
3,f1,LabelA,0.010153,40298880,0.014437,0.023158,0.623386,360,0.010531,0.009878,1.066123,500
4,f1,LabelB,0.006428,40298880,0.014077,0.035236,0.399517,360,0.005963,0.011937,0.499522,500
5,f1,LabelC,0.005008,40298880,0.015016,0.044037,0.340986,360,0.004419,0.013107,0.337161,500
6,f10,LabelA,-0.000335,39383581,0.000350,0.011491,0.030417,360,-0.000328,0.008709,-0.037605,500
7,f10,LabelB,-0.000220,39383581,0.000591,0.017188,0.034409,360,-0.001811,0.007450,-0.243036,500
8,f10,LabelC,-0.000144,39383581,0.001273,0.022312,0.057038,360,-0.002039,0.007085,-0.287815,500
9,f100,LabelA,0.003038,42658752,0.002138,0.008773,0.243717,360,0.004335,0.014097,0.307504,500


In [4]:
# 只读取原始大parquet文件 @data 路径下的前5天数据，不全量加载，且保存为parquet格式（与原始格式保持一致）
import pyarrow.parquet as pq
import pandas as pd
import pyarrow as pa

def get_first_n_dates_from_parquet(path, n=5):
    # 先只读取 dateid 列，找出前 n 个日期
    pf = pq.ParquetFile(path)
    dateid_iter = pf.iter_batches(columns=['dateid'], batch_size=10**6)
    seen_dates = set()
    for batch in dateid_iter:
        arr = batch['dateid'].to_numpy()
        seen_dates.update(arr)
        if len(seen_dates) >= n:
            break
    first_n_dates = sorted(seen_dates)[:n]
    return first_n_dates

def filter_rows_by_dateid_parquet(path, out_path, dateids):
    # 逐批读取，只保存前 n 天的数据，并用parquet存储，格式与原始文件一致
    pf = pq.ParquetFile(path)
    batches = []
    for batch in pf.iter_batches(batch_size=10**6):
        df = batch.to_pandas()
        df_sel = df[df['dateid'].isin(dateids)]
        if not df_sel.empty:
            batches.append(df_sel)
    if batches:
        df_out = pd.concat(batches, ignore_index=True)
        # 保存为parquet（与原始文件相同格式）
        table_out = pa.Table.from_pandas(df_out)
        pq.write_table(table_out, out_path)
        print(f"{out_path} 已保存，共 {len(df_out)} 行。")
    else:
        print(f"未找到 dateid in {dateids} 的数据。")

train_path = "data/train.parquet"
test_path = "data/test.parquet"

first_5_dates_train = get_first_n_dates_from_parquet(train_path, n=5)
filter_rows_by_dateid_parquet(train_path, "train_first5days.parquet", first_5_dates_train)

first_5_dates_test = get_first_n_dates_from_parquet(test_path, n=5)
filter_rows_by_dateid_parquet(test_path, "test_first5days.parquet", first_5_dates_test)

train_first5days.parquet 已保存，共 597500 行。
test_first5days.parquet 已保存，共 597500 行。
